# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a comprehensive walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

The FAIR² dataset contains detailed clinical, pathological, and molecular data for 77 cancer survivors who developed second primary colorectal cancer, including key biomarker and anatomical information, structured using the [Croissant](https://mlcommons.org/croissant/) schema.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD at the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets and their contained fields. All entities are referenced by their Croissant `@id`s.


In [ ]:
# List all available record set IDs and their fields
record_set_objs = list(dataset.record_sets)

print("Available Record Sets:")
for rs in record_set_objs:
    print(f"  - @id: {rs.id} | Name: {rs.name}")

# For each record set, list all fields (columns with Croissant Field @id)
for rs in record_set_objs:
    print(f"\nFields in RecordSet @id={rs.id}:")
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"    - @id: {f.id} | Name: {f.name} | Type: {f.data_type}")
    else:
        print("    (no fields detected)")

## 3. Data Extraction

Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s as discovered in the previous step.

In [ ]:
# Collect all RecordSet @id strings
record_set_ids = [rs.id for rs in dataset.record_sets]

# Extract all data into pandas DataFrames (using @id for reference)
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded RecordSet @id={rs_id} with shape {dataframes[rs_id].shape}")

# Example: Display columns from the main clinical record set (if one exists)
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"\nFields in RecordSet @id={primary_rs_id}:")
    print(dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attributes. All fields referenced use their full Croissant `@id`s as column names in the DataFrame.

In [ ]:
# Choose a record set for EDA (using the first one as example)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Inspect numeric fields by Croissant @id
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
print(f"Selected numeric field for EDA: {numeric_field}")

if numeric_field is not None:
    threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical field if it exists
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'O' and col != numeric_field:
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA in the selected record set.")

## 5. Visualization

Visualize the numeric field distributions or the relationship between key fields using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color="mediumslateblue")
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Optional: Boxplot for the numeric field grouped by the first string/categorical field
    if group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

- The FAIR² dataset provides structured, interoperable clinical and molecular data for 77 survivors with second primary colorectal cancer.
- All record sets, fields, and columns were referenced by their Croissant `@id` for robust, schema-compliant data processing.
- Basic filtering, normalization, grouping, and visualization demonstrate the utility of programmatic Croissant record extraction with `mlcroissant`.
- For deeper analysis, refer to the full Croissant schema and documentation, and cite the original dataset as: 

  > Liu, Y., Duan, X., Yang, S., Zhang, Y. and Han, S. 2026. *Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution*. Frontiers.
